# 📝 TF-IDF Baseline Implementation
## Lexical Retrieval Approach

**Purpose:** Implement and evaluate TF-IDF + Cosine Similarity baseline


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pickle
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Paths
BASE_DIR = "/content/drive/MyDrive/COLIEE_Dataset"
TRAIN_DOCS_DIR = f"{BASE_DIR}/train_docs"
TRAIN_LABELS = f"{BASE_DIR}/labels/task1_train_labels_2025.json"
RESULTS_DIR = f"{BASE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1️⃣ Load Data

In [ ]:
# Load labels
with open(TRAIN_LABELS, 'r') as f:
    train_labels = json.load(f)

# Load all documents
print("Loading documents...")
train_docs = {}
doc_files = sorted(os.listdir(TRAIN_DOCS_DIR))

for i, doc_file in enumerate(doc_files):
    if i % 1000 == 0:
        print(f"  Loaded {i}/{len(doc_files)}...")
    if doc_file.endswith('.txt'):
        with open(os.path.join(TRAIN_DOCS_DIR, doc_file), 'r', encoding='utf-8', errors='ignore') as f:
            train_docs[doc_file] = f.read()

print(f"✓ Loaded {len(train_docs)} documents")

## 2️⃣ Build TF-IDF Model

In [ ]:
print("Building TF-IDF model...")
start_time = time.time()

# Prepare data
doc_ids = list(train_docs.keys())
doc_texts = [train_docs[doc_id] for doc_id in doc_ids]

# Build vectorizer
vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=2,
    max_df=0.8,
    stop_words='english'
)

doc_vectors = vectorizer.fit_transform(doc_texts)

print(f"✓ TF-IDF built in {time.time() - start_time:.2f}s")
print(f"  Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"  Vector shape: {doc_vectors.shape}")

## 3️⃣ Perform Retrieval

In [ ]:
print("Performing retrieval...")
predictions = {}

for i, query_id in enumerate(train_labels.keys()):
    if i % 100 == 0:
        print(f"  Processing {i}/{len(train_labels)}...")
    
    query_text = train_docs[query_id]
    query_vector = vectorizer.transform([query_text])
    
    similarities = cosine_similarity(query_vector, doc_vectors)[0]
    ranked_indices = similarities.argsort()[::-1]
    ranked_doc_ids = [doc_ids[idx] for idx in ranked_indices if doc_ids[idx] != query_id]
    
    predictions[query_id] = ranked_doc_ids

print("✓ Retrieval complete")

## 4️⃣ Evaluate Performance

In [ ]:
def evaluate_retrieval(predictions, labels, k_values=[5, 10, 20]):
    results = {}
    
    for k in k_values:
        precisions = []
        recalls = []
        
        for query_id in predictions:
            if query_id not in labels:
                continue
            
            predicted = set(predictions[query_id][:k])
            relevant = set(labels[query_id])
            
            if len(relevant) == 0:
                continue
            
            tp = len(predicted & relevant)
            precision = tp / k
            recall = tp / len(relevant)
            
            precisions.append(precision)
            recalls.append(recall)
        
        avg_p = np.mean(precisions)
        avg_r = np.mean(recalls)
        f1 = 2 * avg_p * avg_r / (avg_p + avg_r) if (avg_p + avg_r) > 0 else 0
        
        results[f'P@{k}'] = avg_p
        results[f'R@{k}'] = avg_r
        results[f'F1@{k}'] = f1
    
    # Calculate MAP
    aps = []
    for query_id in predictions:
        if query_id not in labels:
            continue
        relevant = set(labels[query_id])
        if len(relevant) == 0:
            continue
        
        predicted = predictions[query_id]
        score = 0.0
        num_hits = 0.0
        
        for i, p in enumerate(predicted):
            if p in relevant:
                num_hits += 1.0
                score += num_hits / (i + 1.0)
        
        if num_hits > 0:
            aps.append(score / len(relevant))
    
    results['MAP'] = np.mean(aps) if aps else 0.0
    return results

# Evaluate
tfidf_results = evaluate_retrieval(predictions, train_labels)

print("\n" + "="*60)
print("TF-IDF RESULTS")
print("="*60)
for metric, value in tfidf_results.items():
    print(f"{metric:8s}: {value:.4f}")
print("="*60)

## 5️⃣ Save Results

In [ ]:
# Save TF-IDF model components
with open(f"{RESULTS_DIR}/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open(f"{RESULTS_DIR}/tfidf_doc_vectors.pkl", "wb") as f:
    pickle.dump(doc_vectors, f)

with open(f"{RESULTS_DIR}/doc_ids.pkl", "wb") as f:
    pickle.dump(doc_ids, f)

# Save results
with open(f"{RESULTS_DIR}/tfidf_results.json", "w") as f:
    json.dump(tfidf_results, f, indent=2)

# Save predictions (for later analysis)
with open(f"{RESULTS_DIR}/tfidf_predictions.json", "w") as f:
    # Save only top-100 per query to save space
    predictions_subset = {k: v[:100] for k, v in predictions.items()}
    json.dump(predictions_subset, f)

print("\n✓ All results saved to:", RESULTS_DIR)

## ✅ TF-IDF Baseline Complete!

Next: Proceed to BERT implementation in notebook 03